In [1]:
import torch
import torch.nn.functional as F
from torch import nn

In [ ]:
# toy example
a = torch.tril(torch.ones(3, 3))
a = a / a.sum(dim = 1, keepdim = True)
b = torch.randint(0, 10, (3,2)).float()
c = a @ b
print('a=')
print(a)
print('----')
print('b=')
print(b)
print('----')
print('c=')
print(c)

In [2]:
# version 4: self-attention
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# single head self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
# let all tokens produce a query and a key (independently)
k = key(x) # (B, T, C) -> (B, T, head_size)
q = query(x) # (B, T, C) -> (B, T, head_size)
v = value(x) # (B, T, C) -> (B, T, head_size)
# let them talk to each other
wei = q @ k.transpose(-1, -2) # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
# scale
wei = wei * head_size ** -0.5 

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float('-inf')) # don't allow tokens to attend to the future
wei = F.softmax(wei, dim=-1)

v = value(x) # (B, T, C) -> (B, T, head_size)
out = wei @ v # (T, T) @ (B, T, C) -> (B, T, C)

out.shape

torch.Size([4, 8, 16])